# Notebook 03. Xây dựng ESG Knowledge Base

## Mục tiêu nghiên cứu

Notebook này có nhiệm vụ xây dựng **ESG Knowledge Base** phục vụ cho quá trình trích xuất đặc trưng ESG từ dữ liệu Amazon trong các bước tiếp theo.

Khác với các notebook trước chỉ xác định **ESG Indicator** và thiết kế **Operational Measurement**, notebook này tập trung xây dựng các nguồn tri thức (Knowledge Base) thông qua việc trích xuất, làm sạch, chuẩn hóa và hợp nhất thông tin ESG từ nhiều nguồn dữ liệu khác nhau.

Các Knowledge Base được tạo ra sẽ được sử dụng trực tiếp trong Notebook 05 để thực hiện Feature Engineering.


## Dữ liệu đầu vào

Notebook sử dụng ba nguồn dữ liệu độc lập.

### ESG Sustainability Reports

Được sử dụng để xây dựng **ESG Keyword Dictionary**.

### Fast Fashion Eco Dataset

Được sử dụng để xây dựng

- Sustainable Material Dictionary
- Eco Label Dictionary

### Amazon Fashion Reviews Dataset (800K)

Được sử dụng để xây dựng **Complaint Dictionary**.

---

## Dữ liệu đầu ra

Notebook tạo ra ESG Knowledge Base tại thư mục

`Dataset/processed/ESG_Knowledge_Base/`

bao gồm

- esg_keyword_dictionary.csv
- sustainable_material_dictionary.csv
- eco_label_dictionary.csv
- complaint_dictionary.csv

Các dictionary này sẽ được sử dụng trực tiếp trong Notebook 05 để xây dựng Product ESG Feature Dataset.

---

# Phần 0. Setup

Chuẩn bị môi trường làm việc cho quá trình xây dựng ESG Knowledge Base.

Trong phần này, notebook sẽ:

- Import các thư viện cần thiết.
- Khai báo đường dẫn dữ liệu đầu vào và đầu ra.
- Kiểm tra sự tồn tại của các tập dữ liệu.
- Đọc các dữ liệu được sử dụng xuyên suốt notebook.

Các tập dữ liệu chuyên biệt sẽ được đọc tại đúng phần xử lý tương ứng nhằm giúp notebook rõ ràng, tiết kiệm bộ nhớ và đúng với luồng nghiên cứu.

In [73]:
# Import thư viện.

from pathlib import Path
from collections import Counter

import re
import string

import numpy as np
import pandas as pd

from IPython.display import display

In [74]:
# Thiết lập đường dẫn thư mục.

project_root = Path.cwd().parent

data_path = (
    project_root / "Data ESG"
)

processed_path = (
    project_root / "Dataset" / "processed"
)

knowledge_base_path = (
    processed_path / "ESG_Knowledge_Base"
)

knowledge_base_path.mkdir(
    parents=True,
    exist_ok=True
)

In [127]:
# Đường dẫn dữ liệu đầu vào.

amazon_reviews_path = (
    data_path
    / "amazon-fashion-800k+-user-reviews-dataset.csv"
)

esg_reports_path = (
    data_path
    / "ESGReport.csv"
)

fast_fashion_company_path = (
    data_path
    / "fastFashionCompDim.csv"
)

fast_fashion_items_path = (
    data_path
    / "fastFasionItemsDim.csv"
)

In [76]:
# Kiểm tra đường dẫn.

path_df = pd.DataFrame(
    {
        "Tên dữ liệu": [
            "Amazon Fashion Reviews Dataset",
            "ESG Sustainability Reports",
            "Fast Fashion Company Dataset",
            "Fast Fashion Items Dataset",
        ],
        "Đường dẫn": [
            amazon_reviews_path,
            esg_reports_path,
            fast_fashion_company_path,
            fast_fashion_items_path,
        ],
    }
)

path_df["Tồn tại"] = (
    path_df["Đường dẫn"]
    .apply(lambda x: x.exists())
)

display(path_df)

,Tên dữ liệu,Đường dẫn,Tồn tại
0,Amazon Fashion Reviews Dataset,c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2...,True
1,ESG Sustainability Reports,c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2...,True
2,Fast Fashion Company Dataset,c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2...,True
3,Fast Fashion Items Dataset,c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2...,True


In [128]:
# Đọc dữ liệu.

amazon_reviews_df = pd.read_csv(
    amazon_reviews_path
)

esg_reports_df = pd.read_csv(
    esg_reports_path
)

In [78]:
# Tổng quan dữ liệu.

dataset_summary = pd.DataFrame(
    {
        "Dataset": [
            "Amazon Fashion Reviews Dataset",
            "ESG Sustainability Reports",
        ],
        "Rows": [
            len(amazon_reviews_df),
            len(esg_reports_df),
        ],
        "Columns": [
            amazon_reviews_df.shape[1],
            esg_reports_df.shape[1],
        ],
    }
)

display(dataset_summary)

,Dataset,Rows,Columns
0,Amazon Fashion Reviews Dataset,867310,11
1,ESG Sustainability Reports,866,10


# Phần 1. Build ESG Keyword Dictionary

Xây dựng **ESG Keyword Dictionary** từ bộ dữ liệu ESG Sustainability Reports.

Dictionary này đóng vai trò là nguồn tri thức (Knowledge Base) để nhận diện các bằng chứng ESG trong dữ liệu Amazon ở Notebook 05.

Quy trình xây dựng gồm các bước:

- Chuẩn bị dữ liệu văn bản.
- Thống kê tần suất xuất hiện của từ khóa.
- Loại bỏ các từ khóa không mang ý nghĩa ESG.
- Gán mỗi từ khóa vào một nhóm ESG.
- Xuất ESG Keyword Dictionary.

In [79]:
# Hiển thị thông tin dữ liệu ESG Reports.

display(
    esg_reports_df.head()
)

print()

esg_reports_df.info()

,Unnamed: 0,filename,ticker,year,preprocessed_content,ner_entities,e_score,s_score,g_score,total_score
0,0,ASX_BSX_2020.pdf,BSX,2020,style guide colour colour use imagecolour prof...,"['bk%', 'rgb', 'un', 'el ectric mine consortiu...",3.16,18.00,11.83,32.98
1,1,ASX_BSX_2022.pdf,BSX,2022,sustainability report look mining green office...,"['murray street', 'west perth', 'west perth', ...",2.83,12.86,10.32,26.02
2,2,ASX_EXR_2022.pdf,EXR,2022,report environment social governance esg basel...,"['september', 'mongolia', 'australia', 'austra...",3.81,4.28,5.86,13.94
3,3,LSE_ADM_2019.pdf,ADM,2019,corporate social responsibilty report introduc...,"['david stevens', 'csr board', 'just over yea...",16.38,14.20,5.90,36.36
4,4,LSE_ADM_2020.pdf,ADM,2020,sustainability admiral commit maintain respons...,"['year', 'health & wellbeing', 'a -month', 'on...",15.89,13.51,5.38,34.78



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 866 entries, 0 to 865
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            866 non-null    int64  
 1   filename              866 non-null    object 
 2   ticker                866 non-null    object 
 3   year                  866 non-null    int64  
 4   preprocessed_content  866 non-null    object 
 5   ner_entities          866 non-null    object 
 6   e_score               866 non-null    float64
 7   s_score               866 non-null    float64
 8   g_score               866 non-null    float64
 9   total_score           866 non-null    float64
dtypes: float64(4), int64(2), object(4)
memory usage: 67.8+ KB


In [80]:
# Chuẩn bị dữ liệu văn bản.

esg_text = (
    esg_reports_df[
        "preprocessed_content"
    ]
    .fillna("")
)

In [81]:
# Tách từ.

tokens = []

for text in esg_text:

    tokens.extend(
        text.split()
    )

print(
    f"Số lượng token: {len(tokens):,}"
)

Số lượng token: 10,361,942


In [82]:
# Thống kê tần suất từ khóa.

from collections import Counter

keyword_frequency = Counter(
    tokens
)

keyword_frequency_df = (
    pd.DataFrame(
        keyword_frequency.items(),
        columns=[
            "keyword",
            "frequency"
        ]
    )
    .sort_values(
        by="frequency",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    keyword_frequency_df.head(20)
)

,keyword,frequency
0,employee,98384
1,include,79028
2,business,72498
3,program,67926
4,risk,64645
5,report,64318
6,work,57378
7,company,56346
8,energy,54235
9,community,53970


### Loại bỏ các từ khóa không mang ý nghĩa ESG

Sau khi thống kê tần suất xuất hiện của từ khóa, nhiều từ phổ biến chỉ mang ý nghĩa mô tả chung của báo cáo doanh nghiệp (ví dụ: *report*, *company*, *include*, *program*, *year*...) sẽ được loại bỏ.

Việc loại bỏ các từ khóa này giúp ESG Keyword Dictionary chỉ giữ lại những từ phản ánh trực tiếp nội dung Environmental, Social và Governance.

In [83]:
# Loại bỏ Generic Words.

generic_words = {

    "report",
    "company",
    "business",
    "include",
    "including",
    "provide",
    "provides",
    "program",
    "work",
    "year",
    "global",
    "product",
    "products",
    "service",
    "services",
    "operation",
    "operations",
    "group",
    "development",
    "performance",
    "organization",
    "support",
    "help",
    "datum",
    "data",
    "new",
    "use",
    "used",
    "using",
    "make",
    "made",
    "well",
    "good",
    "high",
    "low",
    "number",
    "increase",
    "decrease",
    "improve",
    "improved",
    "continue",
    "continued",
    "future",
    "today",
    "time",
    "team",
    "goal",
    "process",
    "base"
}

keyword_frequency_df = (
    keyword_frequency_df[
        ~keyword_frequency_df["keyword"].isin(
            generic_words
        )
    ]
    .reset_index(
        drop=True
    )
)

display(
    keyword_frequency_df.head(20)
)

,keyword,frequency
0,employee,98384
1,risk,64645
2,energy,54235
3,community,53970
4,management,51008
5,emission,49551
6,water,43389
7,customer,42433
8,impact,41170
9,sustainability,40167


### Gán nhóm ESG cho từ khóa

Sau khi loại bỏ các từ khóa phổ biến nhưng không mang ý nghĩa ESG, các từ khóa còn lại được phân loại theo ba nhóm chính:

- Environmental: các thuật ngữ liên quan đến môi trường, tài nguyên, khí hậu, phát thải và sử dụng năng lượng.
- Social: các thuật ngữ liên quan đến con người, người lao động, khách hàng, cộng đồng và an toàn.
- Governance: các thuật ngữ liên quan đến quản trị, rủi ro, chính sách, minh bạch và trách nhiệm doanh nghiệp.

Một số từ khóa có thể xuất hiện trong nhiều ngữ cảnh ESG khác nhau sẽ được gán nhãn ESG_Mixed để tránh áp đặt sai chiều ESG.

In [84]:
# ESG Vocabulary Mapping

environmental_keywords = {

    "energy",
    "emission",
    "water",
    "environmental",
    "climate",
    "sustainability",
    "waste",
    "carbon",
    "renewable",
    "resource",
    "biodiversity",
    "pollution",
    "recycle",
    "recycling",
    "material",
    "reduce"

}


social_keywords = {

    "employee",
    "community",
    "customer",
    "safety",
    "health",
    "supplier",
    "worker",
    "training",
    "diversity",
    "inclusion",
    "human",
    "labor",
    "wellbeing",
    "education"

}


governance_keywords = {

    "risk",
    "management",
    "policy",
    "corporate",
    "governance",
    "board",
    "compliance",
    "audit",
    "ethics",
    "strategy",
    "responsibility",
    "accountability",
    "transparency",
    "opportunity"

}


mixed_keywords = {

    "impact",
    "goal",
    "performance"

}

In [85]:
# Hàm gán ESG Dimension.

def assign_esg_dimension(keyword):

    if keyword in environmental_keywords:
        return "Environmental"

    elif keyword in social_keywords:
        return "Social"

    elif keyword in governance_keywords:
        return "Governance"

    elif keyword in mixed_keywords:
        return "ESG_Mixed"

    else:
        return None

In [86]:
# Gán ESG Dimension.

keyword_frequency_df["ESG_dimension"] = (
    keyword_frequency_df["keyword"]
    .apply(assign_esg_dimension)
)

In [87]:
# Chỉ giữ keyword có ESG Dimension.

esg_keyword_dictionary_df = (
    keyword_frequency_df[
        keyword_frequency_df["ESG_dimension"]
        .notna()
    ]
    .copy()
)

In [88]:
# Thêm nguồn dữ liệu.

esg_keyword_dictionary_df["source"] = (
    "ESG Sustainability Reports"
)

In [89]:
# Kiểm tra ESG Keyword Dictionary.

display(
    esg_keyword_dictionary_df.head(20)
)


print(
    f"Số lượng ESG keywords: "
    f"{len(esg_keyword_dictionary_df):,}"
)

,keyword,frequency,ESG_dimension,source
0,employee,98384,Social,ESG Sustainability Reports
1,risk,64645,Governance,ESG Sustainability Reports
2,energy,54235,Environmental,ESG Sustainability Reports
3,community,53970,Social,ESG Sustainability Reports
4,management,51008,Governance,ESG Sustainability Reports
5,emission,49551,Environmental,ESG Sustainability Reports
6,water,43389,Environmental,ESG Sustainability Reports
7,customer,42433,Social,ESG Sustainability Reports
8,impact,41170,ESG_Mixed,ESG Sustainability Reports
9,sustainability,40167,Environmental,ESG Sustainability Reports


Số lượng ESG keywords: 45


In [90]:
# Phân bố ESG Dimension.

display(
    esg_keyword_dictionary_df[
        "ESG_dimension"
    ]
    .value_counts()
    .reset_index()
)

,ESG_dimension,count
0,Environmental,16
1,Social,14
2,Governance,14
3,ESG_Mixed,1


In [91]:
# Chuẩn hóa cấu trúc output.

esg_keyword_dictionary_df = (
    esg_keyword_dictionary_df[
        [
            "keyword",
            "ESG_dimension",
            "frequency",
            "source"
        ]
    ]
    .sort_values(
        by="frequency",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

In [92]:
# Chuẩn hóa cấu trúc output.

esg_keyword_dictionary_df = (
    esg_keyword_dictionary_df[
        [
            "keyword",
            "ESG_dimension",
            "frequency",
            "source"
        ]
    ]
    .sort_values(
        by="frequency",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    esg_keyword_dictionary_df.head(20)
)

,keyword,ESG_dimension,frequency,source
0,employee,Social,98384,ESG Sustainability Reports
1,risk,Governance,64645,ESG Sustainability Reports
2,energy,Environmental,54235,ESG Sustainability Reports
3,community,Social,53970,ESG Sustainability Reports
4,management,Governance,51008,ESG Sustainability Reports
5,emission,Environmental,49551,ESG Sustainability Reports
6,water,Environmental,43389,ESG Sustainability Reports
7,customer,Social,42433,ESG Sustainability Reports
8,impact,ESG_Mixed,41170,ESG Sustainability Reports
9,sustainability,Environmental,40167,ESG Sustainability Reports


In [93]:
# Đường dẫn output.

esg_keyword_dictionary_output_path = (
    knowledge_base_path
    / "esg_keyword_dictionary.csv"
)

In [94]:
# Export ESG Keyword Dictionary.

esg_keyword_dictionary_df.to_csv(
    esg_keyword_dictionary_output_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Đã lưu ESG Keyword Dictionary:"
)

# Kiểm tra file output.

print(
    esg_keyword_dictionary_output_path.exists()
)

print(
    esg_keyword_dictionary_output_path
)

Đã lưu ESG Keyword Dictionary:
True
c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\processed\ESG_Knowledge_Base\esg_keyword_dictionary.csv


# Phần 2. Build Sustainable Material Dictionary

Trong bước này, notebook khai thác dữ liệu từ Fast Fashion Eco Dataset để xây dựng bộ từ điển vật liệu phục vụ đo lường ESG.

Mục tiêu:

- Xác định các loại vật liệu xuất hiện trong sản phẩm thời trang.
- Chuẩn hóa tên vật liệu theo thuật ngữ ESG quốc tế.
- Gom nhóm các biến thể ngôn ngữ và từ đồng nghĩa.
- Tính tần suất xuất hiện của từng loại vật liệu.
- Xuất bộ từ điển vật liệu bền vững để sử dụng trong bước Feature Engineering.


In [105]:
# Output path - Sustainable Material Dictionary

sustainable_material_dictionary_output_path = (
    knowledge_base_path
    / "sustainable_material_dictionary.csv"
)


print(
    sustainable_material_dictionary_output_path
)

c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\processed\ESG_Knowledge_Base\sustainable_material_dictionary.csv


In [96]:
# Load Sustainable Material Dataset

fast_fashion_material_path = (
    data_path
    / "fastFashionCompDim.csv"
)


fast_fashion_material_df = pd.read_csv(
    fast_fashion_material_path,
    sep="|",
    encoding="utf-8"
)


display(
    fast_fashion_material_df.head()
)


print(
    "Dataset shape:",
    fast_fashion_material_df.shape
)


print(
    fast_fashion_material_df.columns.tolist()
)

,item_code,part_name,material,percent
0,200000,EXTERIOR,algodon,100%
1,200001,EXTERIOR,algodon,100%
2,200002,EXTERIOR,viscosa,62%
3,200002,EXTERIOR,fibra metalizada,37%
4,200002,EXTERIOR,elastano,1%


Dataset shape: (457, 4)
['item_code', 'part_name', 'material', 'percent']


In [101]:
fast_fashion_material_df["material_clean"] = (
    fast_fashion_material_df["material"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

In [102]:
display(
    fast_fashion_material_df[
        [
            "material",
            "material_clean"
        ]
    ].head(10)
)

,material,material_clean
0,algodon,algodon
1,algodon,algodon
2,viscosa,viscosa
3,fibra metalizada,fibra metalizada
4,elastano,elastano
5,viscosa,viscosa
6,poliester,poliester
7,poliester,poliester
8,poliester,poliester
9,poliester,poliester


In [106]:
material_normalization_map = {

    # Cotton
    "algodon": "cotton",
    "algodón": "cotton",

    # Polyester
    "poliester": "polyester",
    "poliéster": "polyester",

    # Elastane
    "elastano": "elastane",

    # Viscose
    "viscosa": "viscose",

    # Polyamide / Nylon
    "poliamida": "polyamide",
    "nailon": "nylon",

    # Lyocell
    "liocel": "lyocell",
    "lyocell": "lyocell",
    "tencel": "lyocell",

    # Linen
    "lino": "linen",

    # Acrylic
    "acrilico": "acrylic",

    # Metallic
    "fibra metalizada": "metallic fiber",

    # Wool
    "lana": "wool",

    # Modal
    "modal": "modal",

    # Other
    "camello": "camel hair",
    "cupro": "cupro"
}


fast_fashion_material_df[
    "material_normalized"
] = (
    fast_fashion_material_df[
        "material_clean"
    ]
    .map(material_normalization_map)
    .fillna(
        fast_fashion_material_df[
            "material_clean"
        ]
    )
)

In [107]:
# Tổng hợp tần suất xuất hiện của từng vật liệu

material_dictionary_df = (
    fast_fashion_material_df[
        "material_normalized"
    ]
    .value_counts()
    .reset_index()
)

material_dictionary_df.columns = [
    "material",
    "frequency"
]

display(
    material_dictionary_df.head(20)
)

,material,frequency
0,polyester,148
1,elastane,91
2,viscose,86
3,cotton,76
4,polyamide,15
5,lyocell,12
6,nylon,10
7,linen,5
8,acrylic,4
9,metallic fiber,3


In [108]:
# Thêm thông tin nguồn dữ liệu

material_dictionary_df["source"] = (
    "Fast Fashion Material Composition Dataset"
)

display(
    material_dictionary_df.head(20)
)

,material,frequency,source
0,polyester,148,Fast Fashion Material Composition Dataset
1,elastane,91,Fast Fashion Material Composition Dataset
2,viscose,86,Fast Fashion Material Composition Dataset
3,cotton,76,Fast Fashion Material Composition Dataset
4,polyamide,15,Fast Fashion Material Composition Dataset
5,lyocell,12,Fast Fashion Material Composition Dataset
6,nylon,10,Fast Fashion Material Composition Dataset
7,linen,5,Fast Fashion Material Composition Dataset
8,acrylic,4,Fast Fashion Material Composition Dataset
9,metallic fiber,3,Fast Fashion Material Composition Dataset


In [109]:
sustainable_material_dictionary_path = (
    knowledge_base_path
    / "sustainable_material_dictionary.csv"
)

material_dictionary_df.to_csv(
    sustainable_material_dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    sustainable_material_dictionary_path
)

Saved: c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\processed\ESG_Knowledge_Base\sustainable_material_dictionary.csv


In [110]:
check_material_dictionary = pd.read_csv(
    sustainable_material_dictionary_path,
    encoding="utf-8-sig"
)

display(
    check_material_dictionary.head(20)
)

print(
    "Shape:",
    check_material_dictionary.shape
)

,material,frequency,source
0,polyester,148,Fast Fashion Material Composition Dataset
1,elastane,91,Fast Fashion Material Composition Dataset
2,viscose,86,Fast Fashion Material Composition Dataset
3,cotton,76,Fast Fashion Material Composition Dataset
4,polyamide,15,Fast Fashion Material Composition Dataset
5,lyocell,12,Fast Fashion Material Composition Dataset
6,nylon,10,Fast Fashion Material Composition Dataset
7,linen,5,Fast Fashion Material Composition Dataset
8,acrylic,4,Fast Fashion Material Composition Dataset
9,metallic fiber,3,Fast Fashion Material Composition Dataset


Shape: (14, 3)


# Phần 3. Xây dựng Eco Label Dictionary

Trong bước này, notebook khai thác Fast Fashion Eco Dataset để xây dựng bộ từ điển các nhãn và bằng chứng liên quan đến tính bền vững trong ngành thời trang.

Mục tiêu:

- Xác định các eco label và sustainability claim xuất hiện trong dữ liệu sản phẩm.
- Chuẩn hóa cách biểu diễn của các nhãn.
- Gộp các biến thể từ ngữ tương đồng.
- Tính tần suất xuất hiện của từng eco label.
- Xuất dictionary phục vụ bước ESG Feature Engineering.

In [111]:
# Output path - Eco Label Dictionary

eco_label_dictionary_output_path = (
    knowledge_base_path
    / "eco_label_dictionary.csv"
)


print(
    eco_label_dictionary_output_path
)

c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\processed\ESG_Knowledge_Base\eco_label_dictionary.csv


In [113]:
# Load Fast Fashion Eco Label Dataset

fast_fashion_eco_path = (
    data_path
    / "fastFasionItemsDim.csv"
)


fast_fashion_eco_df = pd.read_csv(
    fast_fashion_eco_path,
    sep="|",
    encoding="utf-8",
    quotechar='"',
    engine="python"
)


display(
    fast_fashion_eco_df.head()
)


print(
    "Dataset shape:",
    fast_fashion_eco_df.shape
)


print(
    fast_fashion_eco_df.columns.tolist()
)

,item_code,item_name,item_desc,join_life,joinlife_title,joinlife_desc,item_price
0,200000,CAMISA POPELÍN,"""Camisa de cuello solapa y escote pico. Manga ...",True,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ...",1995
1,200001,CAMISA POPELÍN,"""Camisa de cuello solapa y escote pico. Manga ...",True,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ...",1995
2,200002,BLUSA HILO METALIZADO,"""Blusa semitransparente de cuello solapa y esc...",False,NaN,NaN,3995
3,200003,BLUSA SATINADA ALAMARES,"""Blusa de cuello subido y escote pico. Manga l...",False,NaN,NaN,2995
4,200004,BLUSA ESTAMPADA CROPPED,"""Blusa satinada de cuello solapa y manga larga...",False,NaN,NaN,1995


Dataset shape: (276, 7)
['item_code', 'item_name', 'item_desc', 'join_life', 'joinlife_title', 'joinlife_desc', 'item_price']


In [114]:
eco_columns = [
    "join_life",
    "joinlife_title",
    "joinlife_desc"
]

display(
    fast_fashion_eco_df[
        eco_columns
    ].head(10)
)

,join_life,joinlife_title,joinlife_desc
0,True,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ..."
1,True,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ..."
2,False,NaN,NaN
3,False,NaN,NaN
4,False,NaN,NaN
5,False,NaN,NaN
6,False,NaN,NaN
7,False,NaN,NaN
8,True,JOIN LIFE Care for fiber: 100% lino de cultivo...,"""Este lino se cultiva de forma natural, sin ri..."
9,True,JOIN LIFE Care for fiber: 100% lino de cultivo...,"""Este lino se cultiva de forma natural, sin ri..."


In [115]:
joinlife_df = (
    fast_fashion_eco_df[
        fast_fashion_eco_df["join_life"] == True
    ]
    .copy()
)

print(
    f"Số sản phẩm Join Life: {len(joinlife_df)}"
)

display(
    joinlife_df[
        [
            "item_code",
            "joinlife_title",
            "joinlife_desc"
        ]
    ].head(10)
)

Số sản phẩm Join Life: 94


,item_code,joinlife_title,joinlife_desc
0,200000,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ..."
1,200001,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ..."
8,200008,JOIN LIFE Care for fiber: 100% lino de cultivo...,"""Este lino se cultiva de forma natural, sin ri..."
9,200009,JOIN LIFE Care for fiber: 100% lino de cultivo...,"""Este lino se cultiva de forma natural, sin ri..."
10,200010,JOIN LIFE Care for fiber: 100% lino de cultivo...,"""Este lino se cultiva de forma natural, sin ri..."
17,200017,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ..."
18,200018,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ..."
21,200021,JOIN LIFE Care for fiber: 100% algodon organico.,"""Algodon cultivado utilizando fertilizantes y ..."
33,200033,JOIN LIFE Care for fiber: al menos 30% TENCEL™...,"""Esta fibra se obtiene de madera, principalmen..."
34,200034,JOIN LIFE Care for fiber: al menos 75% TENCEL™...,"""Esta fibra se obtiene de madera, principalmen..."


In [116]:
joinlife_title_frequency = (
    joinlife_df["joinlife_title"]
    .value_counts()
    .reset_index()
)

joinlife_title_frequency.columns = [
    "joinlife_title",
    "frequency"
]

display(
    joinlife_title_frequency
)

,joinlife_title,frequency
0,JOIN LIFE Care for fiber: al menos 95% algodon...,19
1,JOIN LIFE Care for fiber: al menos 25% poliest...,14
2,JOIN LIFE Care for fiber: 100% algodon organico.,9
3,JOIN LIFE Care for fiber: al menos 50% algodon...,9
4,JOIN LIFE Care for fiber: al menos 90% algodon...,7
5,JOIN LIFE Care for fiber: al menos 50% viscosa...,7
6,JOIN LIFE Care for water: producido utilizando...,5
7,JOIN LIFE Care for fiber: al menos 30% TENCEL™...,4
8,JOIN LIFE Care for fiber: al menos 50% poliest...,3
9,JOIN LIFE Care for fiber: al menos 75% algodon...,3


In [117]:
joinlife_title_frequency["eco_label_raw"] = (
    joinlife_title_frequency["joinlife_title"]
    .str.replace(
        "JOIN LIFE Care for fiber:",
        "",
        regex=False
    )
    .str.replace(
        "JOIN LIFE Care for water:",
        "",
        regex=False
    )
    .str.strip()
)

display(
    joinlife_title_frequency[
        [
            "joinlife_title",
            "eco_label_raw",
            "frequency"
        ]
    ]
)

,joinlife_title,eco_label_raw,frequency
0,JOIN LIFE Care for fiber: al menos 95% algodon...,al menos 95% algodon organico.,19
1,JOIN LIFE Care for fiber: al menos 25% poliest...,al menos 25% poliester reciclado.,14
2,JOIN LIFE Care for fiber: 100% algodon organico.,100% algodon organico.,9
3,JOIN LIFE Care for fiber: al menos 50% algodon...,al menos 50% algodon organico.,9
4,JOIN LIFE Care for fiber: al menos 90% algodon...,al menos 90% algodon organico.,7
5,JOIN LIFE Care for fiber: al menos 50% viscosa...,al menos 50% viscosa Join Life.,7
6,JOIN LIFE Care for water: producido utilizando...,producido utilizando menos agua.,5
7,JOIN LIFE Care for fiber: al menos 30% TENCEL™...,al menos 30% TENCEL™ Lyocell.,4
8,JOIN LIFE Care for fiber: al menos 50% poliest...,al menos 50% poliester reciclado.,3
9,JOIN LIFE Care for fiber: al menos 75% algodon...,al menos 75% algodon organico & 15% algodon re...,3


In [118]:
eco_label_map = {

    "algodon organico":
        "organic cotton",

    "poliester reciclado":
        "recycled polyester",

    "algodon reciclado":
        "recycled cotton",

    "viscosa ecovero":
        "ecovero viscose",

    "ecovero™ viscose":
        "ecovero viscose",

    "tencel™ lyocell":
        "lyocell",

    "lyocell":
        "lyocell",

    "lino de cultivo europeo":
        "european linen",

    "care for water":
        "water saving"
}

In [121]:
# Chuẩn hóa eco label

import re

def normalize_eco_label(label):

    if pd.isna(label):
        return ""

    label = str(label).lower()

    label = re.sub(
        r"al menos\s*\d+%",
        "",
        label
    )

    label = re.sub(
        r"\d+%",
        "",
        label
    )

    label = (
        label
        .replace(".", "")
        .replace(",", "")
        .strip()
    )

    return label

joinlife_title_frequency[
    "eco_label_clean"
] = (
    joinlife_title_frequency[
        "eco_label_raw"
    ]
    .apply(normalize_eco_label)
)

display(
    joinlife_title_frequency[
        [
            "eco_label_raw",
            "eco_label_clean"
        ]
    ]
)

,eco_label_raw,eco_label_clean
0,al menos 95% algodon organico.,algodon organico
1,al menos 25% poliester reciclado.,poliester reciclado
2,100% algodon organico.,algodon organico
3,al menos 50% algodon organico.,algodon organico
4,al menos 90% algodon organico.,algodon organico
5,al menos 50% viscosa Join Life.,viscosa join life
6,producido utilizando menos agua.,producido utilizando menos agua
7,al menos 30% TENCEL™ Lyocell.,tencel™ lyocell
8,al menos 50% poliester reciclado.,poliester reciclado
9,al menos 75% algodon organico & 15% algodon re...,algodon organico & algodon reciclado


In [123]:
eco_label_translation_map = {

    "algodon organico":
        "organic cotton",

    "poliester reciclado":
        "recycled polyester",

    "algodon reciclado":
        "recycled cotton",

    "tencel™ lyocell":
        "lyocell",

    "tencel lyocell":
        "lyocell",

    "ecovero™ viscose":
        "ecovero viscose",

    "ecovero viscose":
        "ecovero viscose",

    "lino de cultivo europeo":
        "european linen",

    "producido utilizando tecnologias que reducen el consumo de agua":
        "water saving production",
        
    "viscosa join life":
        "join life viscose",

    "producido utilizando menos agua":
        "water saving production",

    "poliamida reciclada":
        "recycled polyamide"
}

joinlife_title_frequency[
    "eco_label"
] = (
    joinlife_title_frequency[
        "eco_label_clean"
    ]
    .map(eco_label_translation_map)
    .fillna(
        joinlife_title_frequency[
            "eco_label_clean"
        ]
    )
)

display(
    joinlife_title_frequency[
        [
            "eco_label_clean",
            "eco_label",
            "frequency"
        ]
    ]
)

,eco_label_clean,eco_label,frequency
0,algodon organico,organic cotton,19
1,poliester reciclado,recycled polyester,14
2,algodon organico,organic cotton,9
3,algodon organico,organic cotton,9
4,algodon organico,organic cotton,7
5,viscosa join life,join life viscose,7
6,producido utilizando menos agua,water saving production,5
7,tencel™ lyocell,lyocell,4
8,poliester reciclado,recycled polyester,3
9,algodon organico & algodon reciclado,algodon organico & algodon reciclado,3


In [124]:
rows = []

for _, row in joinlife_title_frequency.iterrows():

    labels = row["eco_label_clean"].split("&")

    for label in labels:

        label = label.strip()

        label = eco_label_translation_map.get(
            label,
            label
        )

        rows.append({
            "eco_label": label,
            "frequency": row["frequency"]
        })

eco_label_dictionary_df = (
    pd.DataFrame(rows)
    .groupby("eco_label", as_index=False)["frequency"]
    .sum()
    .sort_values(
        "frequency",
        ascending=False
    )
)

eco_label_dictionary_df["source"] = (
    "Fast Fashion Eco Label Dataset"
)

display(eco_label_dictionary_df)

,eco_label,frequency,source
4,organic cotton,50,Fast Fashion Eco Label Dataset
7,recycled polyester,17,Fast Fashion Eco Label Dataset
3,lyocell,8,Fast Fashion Eco Label Dataset
2,join life viscose,7,Fast Fashion Eco Label Dataset
8,water saving production,5,Fast Fashion Eco Label Dataset
1,european linen,3,Fast Fashion Eco Label Dataset
5,recycled cotton,3,Fast Fashion Eco Label Dataset
0,ecovero viscose,2,Fast Fashion Eco Label Dataset
6,recycled polyamide,2,Fast Fashion Eco Label Dataset


In [125]:
eco_label_dictionary_path = (
    knowledge_base_path
    / "eco_label_dictionary.csv"
)

eco_label_dictionary_df.to_csv(
    eco_label_dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    eco_label_dictionary_path
)

Saved: c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\processed\ESG_Knowledge_Base\eco_label_dictionary.csv


In [126]:
check_eco_label_dictionary = pd.read_csv(
    eco_label_dictionary_path,
    encoding="utf-8-sig"
)

display(
    check_eco_label_dictionary
)

print(
    "Shape:",
    check_eco_label_dictionary.shape
)

,eco_label,frequency,source
0,organic cotton,50,Fast Fashion Eco Label Dataset
1,recycled polyester,17,Fast Fashion Eco Label Dataset
2,lyocell,8,Fast Fashion Eco Label Dataset
3,join life viscose,7,Fast Fashion Eco Label Dataset
4,water saving production,5,Fast Fashion Eco Label Dataset
5,european linen,3,Fast Fashion Eco Label Dataset
6,recycled cotton,3,Fast Fashion Eco Label Dataset
7,ecovero viscose,2,Fast Fashion Eco Label Dataset
8,recycled polyamide,2,Fast Fashion Eco Label Dataset


Shape: (9, 3)


# Phần 4. Xây dựng Fashion ESG Complaint Dictionary

Xây dựng **Fashion ESG Complaint Dictionary** từ tập dữ liệu đánh giá Amazon Fashion nhằm phát hiện các nhóm khiếu nại liên quan đến ESG trong đánh giá của khách hàng.

Dictionary này sẽ được sử dụng trong **Notebook 05 - ESG Feature Engineering** để tạo các đặc trưng phản ánh rủi ro ESG của sản phẩm, chẳng hạn:

- Product Safety Complaint
- Counterfeit Complaint
- Packaging Complaint
- Customer Service Complaint
- Product Quality Complaint

Sau phần này sẽ xây dựng được **Fashion ESG Complaint Dictionary**, đóng vai trò là nguồn tri thức (Knowledge Base) phục vụ bước tạo đặc trưng ESG trên dữ liệu Amazon ở Notebook 05.

In [130]:
# Đọc Amazon Fashion Reviews Dataset

amazon_review_path = (
    data_path
    / "amazon-fashion-800k+-user-reviews-dataset.csv"
)

amazon_review_df = pd.read_csv(
    amazon_review_path,
    encoding="utf-8"
)

display(
    amazon_review_df.head()
)

print(
    "Dataset shape:",
    amazon_review_df.shape
)

print(
    amazon_review_df.columns.tolist()
)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchases,target
0,1.0,It say 5 pair when purchasing but only get 2 r...,I was looking for 5 pair and only received 2 p...,[],B07QFTMTLP,B07QFTMTLP,AHASEZ65RESN57BMGRV6QBM5DTIA,1565088068852,0,True,-1
1,1.0,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after...,[],B0764KKDN1,B0764KKDN1,AE3AMA3QSOHFKV46JJAHTHMMIR6A,1622416429592,0,True,-1
2,1.0,Small,Retuned is too small for me,[],B07J1WHVCP,B07J1WHVCP,AH4CFWQE2HTC5BSWIEF3LVLUFK6A,1565284666220,0,True,-1
3,1.0,Pre-Used When Received,This product came with the sleeves turned insi...,[],B0773JWP64,B0773JWP64,AFEKQFJWST6MVTKEJBQKUUBTWK7A,1581963636172,0,False,-1
4,1.0,Worn once and several places at seams have com...,Worn once and several places at seams have com...,[],B099NST9RX,B08JGNS1NK,AGU2FPKN6ARXUSSGBT6WTVLZKJSQ,1640895438476,0,True,-1


Dataset shape: (867310, 11)
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchases', 'target']


In [131]:
# Chỉ giữ các cột phục vụ xây dựng Complaint Dictionary

review_df = amazon_review_df[
    [
        "rating",
        "title",
        "text",
        "target"
    ]
].copy()


# Ghép tiêu đề và nội dung đánh giá

review_df["review_text"] = (
    review_df["title"].fillna("")
    + " "
    + review_df["text"].fillna("")
)


display(
    review_df.head()
)


print(
    f"Số lượng review: {len(review_df):,}"
)

,rating,title,text,target,review_text
0,1.0,It say 5 pair when purchasing but only get 2 r...,I was looking for 5 pair and only received 2 p...,-1,It say 5 pair when purchasing but only get 2 r...
1,1.0,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after...,-1,DonÃ¢ÂÂt do it! Just donÃ¢ÂÂt. These thing...
2,1.0,Small,Retuned is too small for me,-1,Small Retuned is too small for me
3,1.0,Pre-Used When Received,This product came with the sleeves turned insi...,-1,Pre-Used When Received This product came with ...
4,1.0,Worn once and several places at seams have com...,Worn once and several places at seams have com...,-1,Worn once and several places at seams have com...


Số lượng review: 867,310


In [132]:
review_df = amazon_review_df[
    [
        "rating",
        "title",
        "text",
        "target"
    ]
].copy()

review_df["review_text"] = (
    review_df["title"].fillna("")
    + " "
    + review_df["text"].fillna("")
)

display(review_df.head())

print(f"Số lượng review: {len(review_df):,}")

,rating,title,text,target,review_text
0,1.0,It say 5 pair when purchasing but only get 2 r...,I was looking for 5 pair and only received 2 p...,-1,It say 5 pair when purchasing but only get 2 r...
1,1.0,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after...,-1,DonÃ¢ÂÂt do it! Just donÃ¢ÂÂt. These thing...
2,1.0,Small,Retuned is too small for me,-1,Small Retuned is too small for me
3,1.0,Pre-Used When Received,This product came with the sleeves turned insi...,-1,Pre-Used When Received This product came with ...
4,1.0,Worn once and several places at seams have com...,Worn once and several places at seams have com...,-1,Worn once and several places at seams have com...


Số lượng review: 867,310


In [134]:
# Chỉ sử dụng các đánh giá tiêu cực

negative_review_df = (
    review_df[
        review_df["target"] == -1
    ]
    .copy()
)

display(
    negative_review_df.head()
)

print(
    f"Số lượng Negative Reviews: {len(negative_review_df):,}"
)

,rating,title,text,target,review_text
0,1.0,It say 5 pair when purchasing but only get 2 r...,I was looking for 5 pair and only received 2 p...,-1,It say 5 pair when purchasing but only get 2 r...
1,1.0,DonÃ¢ÂÂt do it!,Just donÃ¢ÂÂt. These things fell apart after...,-1,DonÃ¢ÂÂt do it! Just donÃ¢ÂÂt. These thing...
2,1.0,Small,Retuned is too small for me,-1,Small Retuned is too small for me
3,1.0,Pre-Used When Received,This product came with the sleeves turned insi...,-1,Pre-Used When Received This product came with ...
4,1.0,Worn once and several places at seams have com...,Worn once and several places at seams have com...,-1,Worn once and several places at seams have com...


Số lượng Negative Reviews: 346,924


In [135]:
import re

negative_review_df["review_text_clean"] = (
    negative_review_df["review_text"]
    .str.lower()
    .str.replace(
        r"[^a-zA-Z\s]",
        " ",
        regex=True
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

display(
    negative_review_df[
        [
            "review_text",
            "review_text_clean"
        ]
    ].head()
)

,review_text,review_text_clean
0,It say 5 pair when purchasing but only get 2 r...,it say pair when purchasing but only get rip o...
1,DonÃ¢ÂÂt do it! Just donÃ¢ÂÂt. These thing...,don t do it just don t these things fell apart...
2,Small Retuned is too small for me,small retuned is too small for me
3,Pre-Used When Received This product came with ...,pre used when received this product came with ...
4,Worn once and several places at seams have com...,worn once and several places at seams have com...


In [136]:
from nltk.tokenize import word_tokenize

negative_review_df["tokens"] = (
    negative_review_df["review_text_clean"]
    .apply(word_tokenize)
)

display(
    negative_review_df[
        [
            "review_text_clean",
            "tokens"
        ]
    ].head()
)

,review_text_clean,tokens
0,it say pair when purchasing but only get rip o...,"[it, say, pair, when, purchasing, but, only, g..."
1,don t do it just don t these things fell apart...,"[don, t, do, it, just, don, t, these, things, ..."
2,small retuned is too small for me,"[small, retuned, is, too, small, for, me]"
3,pre used when received this product came with ...,"[pre, used, when, received, this, product, cam..."
4,worn once and several places at seams have com...,"[worn, once, and, several, places, at, seams, ..."


In [137]:
from nltk.corpus import stopwords

stop_words = set(
    stopwords.words("english")
)

negative_review_df["tokens"] = (
    negative_review_df["tokens"]
    .apply(
        lambda words: [
            word
            for word in words
            if word not in stop_words
        ]
    )
)

display(
    negative_review_df[
        "tokens"
    ].head()
)

0    [say, pair, purchasing, get, rip, looking, pai...
1    [things, fell, apart, first, wash, uncomfortab...
2                              [small, retuned, small]
3    [pre, used, received, product, came, sleeves, ...
4    [worn, several, places, seams, come, apart, le...
Name: tokens, dtype: object

In [138]:
from nltk.util import bigrams

negative_review_df["bigrams"] = (
    negative_review_df["tokens"]
    .apply(
        lambda words: list(
            bigrams(words)
        )
    )
)

display(
    negative_review_df[
        "bigrams"
    ].head()
)

0    [(say, pair), (pair, purchasing), (purchasing,...
1    [(things, fell), (fell, apart), (apart, first)...
2                 [(small, retuned), (retuned, small)]
3    [(pre, used), (used, received), (received, pro...
4    [(worn, several), (several, places), (places, ...
Name: bigrams, dtype: object

In [139]:
from collections import Counter

bigram_counter = Counter()

for review in negative_review_df["bigrams"]:

    bigram_counter.update(review)

bigram_frequency_df = (
    pd.DataFrame(
        [
            (
                " ".join(bg),
                freq
            )
            for bg, freq in bigram_counter.items()
        ],
        columns=[
            "bigram",
            "frequency"
        ]
    )
    .sort_values(
        "frequency",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    bigram_frequency_df.head(100)
)

,bigram,frequency
0,two stars,24270
1,one star,21504
2,br br,14617
3,poor quality,12726
4,way small,10566
...,...,...
95,fit right,1379
96,size fit,1375
97,really like,1350
98,buyer beware,1339


In [149]:
invalid_bigram = {

    "one star",
    "two stars",
    "three stars",
    "four stars",
    "five stars",
    "br br",
    "years ago",
    "year old",
    "look like",
    "really like",
    "would recommend",
    "fit right"
}

complaint_bigram_df = (
    bigram_frequency_df[
        ~bigram_frequency_df["bigram"].isin(
            invalid_bigram
        )
    ]
    .copy()
)

display(
    complaint_bigram_df.head(100)
)

,bigram,frequency
3,poor quality,12726
4,way small,10566
5,waste money,10531
6,cheaply made,9356
7,like picture,8815
...,...,...
103,super small,1294
104,size way,1268
105,looks great,1252
106,size large,1252


In [154]:
complaint_ontology = {

    # Product Quality

    "poor quality": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    "terrible quality": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    "bad quality": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    "cheap quality": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    "low quality": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    "cheap material": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    "terrible material": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    "poor stitching": (
        "Product Quality",
        "poor quality",
        "S"
    ),

    # Product Damage

    "fell apart": (
        "Product Damage",
        "product damage",
        "S"
    ),

    "fall apart": (
        "Product Damage",
        "product damage",
        "S"
    ),

    "came apart": (
        "Product Damage",
        "product damage",
        "S"
    ),

    "ripped apart": (
        "Product Damage",
        "product damage",
        "S"
    ),

    "broken zipper": (
        "Product Damage",
        "product damage",
        "S"
    ),

    "broken button": (
        "Product Damage",
        "product damage",
        "S"
    ),

    "broken strap": (
        "Product Damage",
        "product damage",
        "S"
    ),

    "torn seam": (
        "Product Damage",
        "product damage",
        "S"
    ),

    # Product Safety

    "chemical smell": (
        "Product Safety",
        "chemical smell",
        "S"
    ),

    "bad smell": (
        "Product Safety",
        "chemical smell",
        "S"
    ),

    "strong smell": (
        "Product Safety",
        "chemical smell",
        "S"
    ),

    "smells bad": (
        "Product Safety",
        "chemical smell",
        "S"
    ),

    "funny smell": (
        "Product Safety",
        "chemical smell",
        "S"
    ),

    # Product Sizing

    "wrong size": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "way small": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "too small": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "too big": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "way big": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "fit small": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "fit large": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "size small": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    "size large": (
        "Product Sizing",
        "wrong size",
        "G"
    ),

    # Counterfeit

    "fake product": (
        "Counterfeit",
        "counterfeit product",
        "G"
    ),

    "buyer beware": (
        "Counterfeit",
        "counterfeit product",
        "G"
    ),

    # Customer Service

    "customer service": (
        "Customer Service",
        "poor customer service",
        "G"
    ),

    # Packaging

    "damaged packaging": (
        "Packaging",
        "packaging damage",
        "E"
    ),

    "waste packaging": (
        "Packaging",
        "packaging damage",
        "E"
    )

}

In [156]:
ontology_rows = []

for _, row in complaint_bigram_df.iterrows():

    phrase = row["bigram"]

    if phrase in complaint_ontology:

        category, normalized, esg = complaint_ontology[phrase]

        ontology_rows.append({

            "raw_complaint": phrase,

            "complaint_category": category,

            "normalized_complaint": normalized,

            "esg_dimension": esg,

            "frequency": row["frequency"]

        })

ontology_df = pd.DataFrame(ontology_rows)

display(
    ontology_df.head(20)
)

,raw_complaint,complaint_category,normalized_complaint,esg_dimension,frequency
0,poor quality,Product Quality,poor quality,S,12726
1,way small,Product Sizing,wrong size,G,10566
2,cheap material,Product Quality,poor quality,S,4383
3,way big,Product Sizing,wrong size,G,4269
4,fell apart,Product Damage,product damage,S,3839
5,size small,Product Sizing,wrong size,G,2983
6,low quality,Product Quality,poor quality,S,2529
7,bad quality,Product Quality,poor quality,S,2254
8,cheap quality,Product Quality,poor quality,S,2066
9,wrong size,Product Sizing,wrong size,G,1508


In [157]:
fashion_esg_complaint_ontology_df = (

    ontology_df

    .groupby(

        [
            "complaint_category",
            "normalized_complaint",
            "esg_dimension"
        ],

        as_index=False

    )["frequency"]

    .sum()

    .sort_values(

        "frequency",

        ascending=False

    )

    .reset_index(drop=True)

)

fashion_esg_complaint_ontology_df["source"] = (
    "Amazon Fashion Reviews Dataset"
)

display(
    fashion_esg_complaint_ontology_df
)

,complaint_category,normalized_complaint,esg_dimension,frequency,source
0,Product Quality,poor quality,S,25746,Amazon Fashion Reviews Dataset
1,Product Sizing,wrong size,G,22047,Amazon Fashion Reviews Dataset
2,Product Damage,product damage,S,6287,Amazon Fashion Reviews Dataset
3,Counterfeit,counterfeit product,G,1367,Amazon Fashion Reviews Dataset
4,Customer Service,poor customer service,G,1094,Amazon Fashion Reviews Dataset
5,Product Safety,chemical smell,S,924,Amazon Fashion Reviews Dataset
6,Packaging,packaging damage,E,11,Amazon Fashion Reviews Dataset


In [158]:
fashion_esg_complaint_ontology_path = (
    knowledge_base_path
    / "fashion_esg_complaint_ontology.csv"
)

fashion_esg_complaint_ontology_df.to_csv(
    fashion_esg_complaint_ontology_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    fashion_esg_complaint_ontology_path
)

Saved: c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\processed\ESG_Knowledge_Base\fashion_esg_complaint_ontology.csv
